# Day 1: Tokenization

A transformer never sees text, only sequences of integer ids. The tokenizer splits text into pieces from a fixed vocabulary and maps every piece to its id. We use the tokenizer of `distilbert-base-uncased`, the model we will fine-tune on day 5.

In [1]:
import pandas as pd
import torch

from itg_transformers.config import MAX_LENGTH, MODEL_NAME, SPLITS
from itg_transformers.data import load_split
from itg_transformers.eda import word_counts
from itg_transformers.tokenization import (
    count_tokens,
    explain_tokenization,
    load_tokenizer,
    tokenize_texts,
)

print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

torch: 2.14.1
MPS available: True


## 1. Loading the tokenizer

In [2]:
tokenizer = load_tokenizer()

print("Model:", MODEL_NAME)
print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("model_max_length:", tokenizer.model_max_length)
print("Our MAX_LENGTH:", MAX_LENGTH)

Model: distilbert-base-uncased
Tokenizer class: BertTokenizer
Vocabulary size: 30522
model_max_length: 512
Our MAX_LENGTH: 64


`model_max_length` is 512: DistilBERT has position embeddings for 512 tokens. Tweets are much shorter, so we use `MAX_LENGTH = 64`, which truncates less than 0.01% of tweets (see `00_eda.ipynb`). Shorter sequences make training faster, since attention cost grows with the square of the length.

## 2. Tokenizing one text

Calling the tokenizer returns a dict with `input_ids` (token ids) and `attention_mask` (1 for every real token).

In [3]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text)
print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [4]:
input_ids = tokens["input_ids"]
print(f"Number of tokens: {len(input_ids)}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(input_ids)}")

Number of tokens: 8
Tokens: ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'amazing', '!', '[SEP]']


`[CLS]` and `[SEP]` are special tokens the tokenizer adds around every text (section 4). The model is *uncased*: the text is lowercased before splitting, so all spellings below become the same token. Shouting in capitals is invisible to this model.

In [5]:
for word in ["movie", "Movie", "MOVIE", " movie"]:
    ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    print(f"{word!r:10} -> {tokenizer.convert_ids_to_tokens(ids)} {ids}")

'movie'    -> ['movie'] [3185]
'Movie'    -> ['movie'] [3185]
'MOVIE'    -> ['movie'] [3185]
' movie'   -> ['movie'] [3185]


Decoding turns the ids back into text:

In [6]:
print(f"Decoded: {tokenizer.decode(input_ids)}")
print(
    "Without special tokens: "
    f"{tokenizer.decode(input_ids, skip_special_tokens=True)}"
)

Decoded: [CLS] this movie was absolutely amazing! [SEP]
Without special tokens: this movie was absolutely amazing!


The same for a real tweet from train. DistilBERT uses WordPiece: a word missing from the vocabulary is split into pieces, and every piece except the first is marked with `##`, meaning "glued to the previous piece".

In [7]:
tweet = load_split("train")["text"].iloc[0]
tweet_ids = tokenizer(tweet)["input_ids"]
decoded_tweet = tokenizer.decode(tweet_ids, skip_special_tokens=True)

print(tweet)
print(f"\n{len(tweet.split())} words -> {len(tweet_ids)} tokens")
print(tokenizer.convert_ids_to_tokens(tweet_ids))
print(f"\nDecoded: {decoded_tweet}")
print(f"Round trip is lossless: {decoded_tweet == tweet}")

QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin

18 words -> 36 tokens
['[CLS]', 'q', '##t', '@', 'user', 'in', 'the', 'original', 'draft', 'of', 'the', '7th', 'book', ',', 're', '##mus', 'lu', '##pin', 'survived', 'the', 'battle', 'of', 'hog', '##wart', '##s', '.', '#', 'happy', '##bir', '##th', '##day', '##rem', '##us', '##lu', '##pin', '[SEP]']

Decoded: qt @ user in the original draft of the 7th book, remus lupin survived the battle of hogwarts. # happybirthdayremuslupin
Round trip is lossless: False


Unlike byte-level BPE, the round trip is lossy: case is gone and spaces around punctuation are restored by heuristics.

## 3. Batches

A model processes many texts at once, as one tensor of shape `(batch_size, seq_len)`. Texts have different lengths, so `tokenize_texts` pads the shorter ones with `[PAD]` up to the longest text in the batch and cuts anything longer than `max_length`.

In [8]:
texts = [
    "This movie was great!",
    "Terrible movie, waste of time.",
]
batch = tokenize_texts(texts, tokenizer)

print(f"Keys: {list(batch.keys())}")
print(f"Shape: {batch['input_ids'].shape}")
print(f"Input IDs:\n{batch['input_ids']}")
print(f"Attention mask:\n{batch['attention_mask']}")

Keys: ['input_ids', 'attention_mask']
Shape: torch.Size([2, 9])
Input IDs:
tensor([[ 101, 2023, 3185, 2001, 2307,  999,  102,    0,    0],
        [ 101, 6659, 3185, 1010, 5949, 1997, 2051, 1012,  102]])
Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [9]:
for ids in batch["input_ids"]:
    print(tokenizer.convert_ids_to_tokens(ids))

['[CLS]', 'this', 'movie', 'was', 'great', '!', '[SEP]', '[PAD]', '[PAD]']
['[CLS]', 'terrible', 'movie', ',', 'waste', 'of', 'time', '.', '[SEP]']


The first text is shorter, so it got two `[PAD]` tokens (id 0) at the end. `attention_mask` has 0 in those positions: it tells the model to ignore them. Without the mask the padding would change the result, and a text would get a different prediction depending on which batch it landed in.

Padding goes to the longest text *in the batch*, not to `max_length`. Batches of short tweets stay short, which saves computation.

Truncation in action, with a deliberately tiny `max_length`:

In [10]:
short = tokenize_texts(texts, tokenizer, max_length=6)
for ids in short["input_ids"]:
    print(tokenizer.convert_ids_to_tokens(ids))

['[CLS]', 'this', 'movie', 'was', 'great', '[SEP]']
['[CLS]', 'terrible', 'movie', ',', 'waste', '[SEP]']


The end of the text is cut, but `[SEP]` is kept: the tokenizer always leaves room for the special tokens.

A real batch of 8 tweets from train:

In [11]:
tweets = load_split("train")["text"].head(8).tolist()
tweet_batch = tokenize_texts(tweets, tokenizer)
mask = tweet_batch["attention_mask"]

print(f"Shape: {tweet_batch['input_ids'].shape}")
print(f"Real tokens per tweet: {mask.sum(dim=1).tolist()}")
print(f"Padding share: {1 - mask.float().mean():.0%}")

Shape: torch.Size([8, 41])
Real tokens per tweet: [36, 22, 29, 29, 30, 30, 41, 20]
Padding share: 28%


## 4. Special tokens

Special tokens are not parts of words. The tokenizer inserts them to give the model structure: where a text starts and ends, which positions are padding, which piece is unknown.

In [12]:
special = {
    "CLS": tokenizer.cls_token,
    "SEP": tokenizer.sep_token,
    "PAD": tokenizer.pad_token,
    "UNK": tokenizer.unk_token,
    "MASK": tokenizer.mask_token,
}
for name, token in special.items():
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"{name:5} token: {token!s:7} (ID: {token_id})")

CLS   token: [CLS]   (ID: 101)
SEP   token: [SEP]   (ID: 102)
PAD   token: [PAD]   (ID: 0)
UNK   token: [UNK]   (ID: 100)
MASK  token: [MASK]  (ID: 103)


- `[CLS]` (*classification*) opens every text. After the model has read the whole text, the vector at this position serves as a summary of it; the classification head on days 4-5 is built on top of it.
- `[SEP]` (*separator*) closes the text. For a pair of texts it also separates them.
- `[PAD]` fills batches up to the same length (section 3).
- `[UNK]` (*unknown*) replaces characters missing from the vocabulary.
- `[MASK]` hides a token during pre-training: the model learned language by guessing masked words. We will not need it.

In [13]:
single = tokenizer(text, return_tensors="pt", return_token_type_ids=False)
print(f"Input IDs: {single['input_ids']}")
print(f"Decoded: {tokenizer.decode(single['input_ids'][0])}")

Input IDs: tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102]])
Decoded: [CLS] this movie was absolutely amazing! [SEP]


`101` and `102` at the edges are `[CLS]` and `[SEP]`. A pair of texts gets one `[SEP]` after each of them:

In [14]:
pair = tokenizer("Is the movie good?", "It was amazing!")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))

['[CLS]', 'is', 'the', 'movie', 'good', '?', '[SEP]', 'it', 'was', 'amazing', '!', '[SEP]']


`[UNK]` matters for our data: the uncased vocabulary has no emoji, and 6.8% of test tweets contain them (see `00_eda.ipynb`). The model cannot tell one emoji from another:

In [15]:
for emoji_text in ["so happy 😍", "so happy 😩", "so happy 🤔"]:
    print(f"{emoji_text!r:15} -> {tokenizer.tokenize(emoji_text)}")

'so happy 😍'    -> ['so', 'happy', '[UNK]']
'so happy 😩'    -> ['so', 'happy', '[UNK]']
'so happy 🤔'    -> ['so', 'happy', '[UNK]']


## 5. `explain_tokenization`

`tokenizer.tokenize` returns the pieces without special tokens, `convert_tokens_to_ids` looks up their ids in the vocabulary.

In [16]:
explain_tokenization("Transformers are amazing!", tokenizer)

Text: Transformers are amazing!
Tokens: ['transformers', 'are', 'amazing', '!']
IDs: [19081, 2024, 6429, 999]
Count: 4


Tweets are full of things a model trained on Wikipedia and books has rarely seen. How the tokenizer handles them:

In [17]:
examples = {
    "hashtag": "#HappyBirthday to my best friend",
    "elongated": "this is sooooo good",
    "shouting": "I HATE MONDAYS",
    "slang": "lol idk tbh smh",
    "mention": "@user thanks!!!",
    "typo": "definately the best",
    "apostrophe": "I can’t wait",
    "emoji": "love it 😍",
}
for name, example in examples.items():
    print(f"--- {name}")
    explain_tokenization(example, tokenizer)

--- hashtag
Text: #HappyBirthday to my best friend
Tokens: ['#', 'happy', '##bir', '##th', '##day', 'to', 'my', 'best', 'friend']
IDs: [1001, 3407, 17706, 2705, 10259, 2000, 2026, 2190, 2767]
Count: 9
--- elongated
Text: this is sooooo good
Tokens: ['this', 'is', 'soo', '##oo', '##o', 'good']
IDs: [2023, 2003, 17111, 9541, 2080, 2204]
Count: 6
--- shouting
Text: I HATE MONDAYS
Tokens: ['i', 'hate', 'mondays']
IDs: [1045, 5223, 28401]
Count: 3
--- slang
Text: lol idk tbh smh
Tokens: ['lo', '##l', 'id', '##k', 'tb', '##h', 'sm', '##h']
IDs: [8840, 2140, 8909, 2243, 26419, 2232, 15488, 2232]
Count: 8
--- mention
Text: @user thanks!!!
Tokens: ['@', 'user', 'thanks', '!', '!', '!']
IDs: [1030, 5310, 4283, 999, 999, 999]
Count: 6
--- typo
Text: definately the best
Tokens: ['def', '##inate', '##ly', 'the', 'best']
IDs: [13366, 14776, 2135, 1996, 2190]
Count: 5
--- apostrophe
Text: I can’t wait
Tokens: ['i', 'can', '’', 't', 'wait']
IDs: [1045, 2064, 1521, 1056, 3524]
Count: 5
--- emoji
Text: 

### How often words get split

Every distinct word of train (lowercased, without stop words, hashtags included) tokenized on its own. *Pieces* is the number of tokens the word turns into.

In [18]:
counts = word_counts(load_split("train")["text"])
words = pd.DataFrame(
    {
        "count": counts,
        "pieces": [len(tokenizer.tokenize(word)) for word in counts.index],
    }
)
split = words["pieces"] > 1
print(f"Distinct words: {len(words)}")
print(f"Split into several pieces: {split.mean():.1%} of distinct words")
print(
    "Split, weighted by frequency: "
    f"{words.loc[split, 'count'].sum() / words['count'].sum():.1%} "
    "of word occurrences"
)

Distinct words: 46719
Split into several pieces: 67.1% of distinct words
Split, weighted by frequency: 19.0% of word occurrences


The most frequent words that are split into pieces. Contractions like `don't` are left out: BERT always splits on punctuation, so `don ' t` is by design, not a missing word. Note that `’` and `'` are different tokens, so `i’m` and `i'm` look different to the model.

In [19]:
has_apostrophe = words.index.str.contains("['’]", regex=True)
frequent_split = words[split & ~has_apostrophe].head(20).copy()
frequent_split["tokens"] = [
    " ".join(tokenizer.tokenize(word)) for word in frequent_split.index
]
frequent_split

,count,pieces,tokens
lol,663.0,2,lo ##l
cena,354.0,2,ce ##na
eid,320.0,2,e ##id
nicki,303.0,2,nick ##i
sharknado,295.0,3,shark ##nad ##o
dunkin,281.0,2,dun ##kin
naruto,276.0,3,na ##rut ##o
gucci,275.0,2,gu ##cci
rousey,271.0,2,rouse ##y
ps4,263.0,2,ps ##4


## 6. Bonus: DistilBERT vs. a Twitter tokenizer

`cardiffnlp/twitter-roberta-base` is RoBERTa further pre-trained on 58M tweets. It keeps the original RoBERTa vocabulary: byte-level BPE, case-sensitive, no `[UNK]`. Same examples, both tokenizers:

In [20]:
tokenizers = {
    "distilbert": tokenizer,
    "twitter-roberta": load_tokenizer("cardiffnlp/twitter-roberta-base"),
}
pd.DataFrame(
    {
        name: [" ".join(tok.tokenize(text)) for text in examples.values()]
        for name, tok in tokenizers.items()
    },
    index=list(examples),
)

,distilbert,twitter-roberta
hashtag,# happy ##bir ##th ##day to my best friend,# Happy Birth day Ġto Ġmy Ġbest Ġfriend
elongated,this is soo ##oo ##o good,this Ġis Ġso oooo Ġgood
shouting,i hate mondays,I ĠH ATE ĠM OND AY S
slang,lo ##l id ##k tb ##h sm ##h,lol Ġid k Ġt bh Ġsm h
mention,@ user thanks ! ! !,@ user Ġthanks !!!
typo,def ##inate ##ly the best,def inately Ġthe Ġbest
apostrophe,i can ’ t wait,I Ġcan âĢ Ļ t Ġwait
emoji,love it [UNK],love Ġit ĠðŁĺ į


RoBERTa marks the *start* of a word with `Ġ` (a space), DistilBERT marks the *continuation* with `##`. Emoji in RoBERTa are split into UTF-8 bytes, which look like `ðŁĺ` but are distinct for every emoji.

Numbers over the whole dataset:

In [21]:
splits = {name: load_split(name)["text"].tolist() for name in SPLITS}
rows = {}
for name, tok in tokenizers.items():
    row = {"vocab_size": tok.vocab_size}
    for split, texts in splits.items():
        row[f"median_tokens_{split}"] = pd.Series(
            count_tokens(tok, texts)
        ).median()
    test_ids = tok(splits["test"])["input_ids"]
    row["test_tweets_with_unk"] = sum(
        tok.unk_token_id in ids for ids in test_ids
    ) / len(test_ids)
    rows[name] = row
pd.DataFrame(rows).T

,vocab_size,median_tokens_train,median_tokens_validation,median_tokens_test,test_tweets_with_unk
distilbert,30522.0,29.0,29.0,26.0,0.068056
twitter-roberta,50265.0,29.0,29.0,26.0,0.000000


Both produce tweets of about the same length, so the difference is not efficiency but what survives tokenization: case, emoji and exact spelling. The tokenizer is not the main advantage of twitter-roberta though; its weights have seen tweets. Day 6 can compare our fine-tuned DistilBERT with a Twitter model.